# 04.1 Project Structure / 项目结构与代码拆分

这一节的目标是把你从“一个 notebook 里全都写完”推进到“可以整理成小项目”。  
The goal of this notebook is to move you from "everything lives in one notebook" to "I can organize this into a small project".

重点概念 / Key concepts:

- 职责分离 / separation of concerns
- 数据模块 / dataset module
- 模型模块 / model module
- 训练引擎 / training engine
- 配置管理 / configuration management

## 学习目标 / Learning Goals

学完后你应该能 / After this notebook, you should be able to:

1. 理解为什么 notebook 原型要进一步拆分 / Understand why a notebook prototype should later be split into modules.
2. 分清 `dataset.py / model.py / engine.py` 的职责 / Distinguish the responsibilities of `dataset.py / model.py / engine.py`.
3. 看懂一个最小可复现项目的目录结构 / Read a minimal reproducible project structure.
4. 把一个 toy 训练流程拆到独立文件里 / Split a toy training workflow into separate files.
5. 明白“先在 notebook 探索，再抽到模块”这条常见路线 / Understand the common workflow of exploring in notebooks first, then extracting reusable code.

In [ ]:
import os
import sys
import tempfile
import textwrap
from pathlib import Path

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

torch.manual_seed(42)

## 1. 典型起点：所有东西都写在一个 notebook 里
## A Typical Starting Point: Everything in One Notebook

这在学习阶段完全正常。  
This is completely normal during the learning stage.

问题通常出在后面 / The problems usually appear later:

- 想复用数据处理逻辑时很难找 / it becomes hard to locate reusable data-processing logic
- 模型定义和训练循环混在一起 / model definition and training loop are mixed together
- 改一个地方容易影响别处 / changing one thing can easily affect something else
- 项目越来越难维护 / the project becomes harder to maintain

In [ ]:
def make_toy_data(n_samples=320):
    x = torch.randn(n_samples, 2)
    y = (x[:, 0] + 0.7 * x[:, 1] > 0).long()
    return x, y


class NotebookPrototypeNet(nn.Module):
    def __init__(self, in_dim=2, hidden_dim=16, num_classes=2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, num_classes),
        )

    def forward(self, x):
        return self.net(x)


def run_epoch_notebook(model, loader, loss_fn, optimizer=None):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()
    total_loss = 0.0
    total_correct = 0
    total_items = 0

    for xb, yb in loader:
        with torch.set_grad_enabled(is_train):
            logits = model(xb)
            loss = loss_fn(logits, yb)

        if is_train:
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        preds = logits.argmax(dim=1)
        total_loss += loss.item() * xb.size(0)
        total_correct += (preds == yb).sum().item()
        total_items += xb.size(0)

    return total_loss / total_items, total_correct / total_items


x, y = make_toy_data()
train_ds = TensorDataset(x[:256], y[:256])
val_ds = TensorDataset(x[256:], y[256:])
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=32, shuffle=False)

model = NotebookPrototypeNet()
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)

for epoch in range(1, 4):
    train_loss, train_acc = run_epoch_notebook(model, train_loader, loss_fn, optimizer=optimizer)
    val_loss, val_acc = run_epoch_notebook(model, val_loader, loss_fn, optimizer=None)
    print(
        f"prototype epoch={epoch:02d} | train_loss={train_loss:.4f} | train_acc={train_acc:.4f} | "
        f"val_loss={val_loss:.4f} | val_acc={val_acc:.4f}"
    )

上面的代码能跑，但职责是混在一起的。  
The code above works, but the responsibilities are mixed together.

接下来我们把它整理成小项目。  
Next, we reorganize it into a small project.

## 2. 一个最小项目如何拆 / How to Split a Minimal Project

先记住一个够实用的分工 / Start with this practical split:

- `dataset.py`：准备数据、返回 `DataLoader` / prepare data and return `DataLoader`
- `model.py`：定义网络结构 / define the model architecture
- `engine.py`：训练和验证逻辑 / training and validation logic
- `config.py`：超参数和运行配置 / hyperparameters and run configuration
- `train.py`：把上面几部分拼起来运行 / wire everything together and run

In [ ]:
project_root = Path(tempfile.mkdtemp(prefix="phase4_project_structure_"))
(project_root / "src").mkdir(parents=True, exist_ok=True)
(project_root / "configs").mkdir(parents=True, exist_ok=True)

files = {
    project_root / "src" / "__init__.py": "",
    project_root / "configs" / "__init__.py": "",
    project_root / "src" / "dataset.py": textwrap.dedent(
        """
        import torch
        from torch.utils.data import DataLoader, TensorDataset

        def make_toy_loaders(batch_size=32, seed=42):
            g = torch.Generator().manual_seed(seed)
            x = torch.randn(320, 2, generator=g)
            y = (x[:, 0] + 0.7 * x[:, 1] > 0).long()

            train_ds = TensorDataset(x[:256], y[:256])
            val_ds = TensorDataset(x[256:], y[256:])

            train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
            val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False)
            return train_loader, val_loader
        """
    ).strip()
    + "\n",
    project_root / "src" / "model.py": textwrap.dedent(
        """
        import torch.nn as nn

        class TinyClassifier(nn.Module):
            def __init__(self, in_dim=2, hidden_dim=16, num_classes=2):
                super().__init__()
                self.net = nn.Sequential(
                    nn.Linear(in_dim, hidden_dim),
                    nn.ReLU(),
                    nn.Linear(hidden_dim, num_classes),
                )

            def forward(self, x):
                return self.net(x)
        """
    ).strip()
    + "\n",
    project_root / "src" / "engine.py": textwrap.dedent(
        """
        import torch

        def run_epoch(model, loader, loss_fn, optimizer=None):
            is_train = optimizer is not None
            model.train() if is_train else model.eval()
            total_loss = 0.0
            total_correct = 0
            total_items = 0

            for xb, yb in loader:
                with torch.set_grad_enabled(is_train):
                    logits = model(xb)
                    loss = loss_fn(logits, yb)

                if is_train:
                    optimizer.zero_grad()
                    loss.backward()
                    optimizer.step()

                preds = logits.argmax(dim=1)
                total_loss += loss.item() * xb.size(0)
                total_correct += (preds == yb).sum().item()
                total_items += xb.size(0)

            return total_loss / total_items, total_correct / total_items
        """
    ).strip()
    + "\n",
    project_root / "configs" / "default_config.py": textwrap.dedent(
        """
        CONFIG = {
            "batch_size": 32,
            "hidden_dim": 16,
            "lr": 0.1,
            "epochs": 5,
            "seed": 42,
        }
        """
    ).strip()
    + "\n",
}

for path, content in files.items():
    path.write_text(content, encoding="utf-8")

for root, dirs, file_names in os.walk(project_root):
    rel_root = Path(root).relative_to(project_root)
    indent = "  " * len(rel_root.parts)
    label = "." if str(rel_root) == "." else rel_root.name
    print(f"{indent}{label}/")
    for file_name in sorted(file_names):
        print(f"{indent}  {file_name}")

print("project_root =", project_root)

上面这个目录还很小，但已经具备了“能扩展”的基本形态。  
This directory is still very small, but it already has the basic shape of something extensible.

你以后加实验、换模型、换数据集时，就不会所有逻辑都堆在 notebook 里。  
When you later add experiments, switch models, or switch datasets, everything will not be piled into one notebook.

In [ ]:
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from configs.default_config import CONFIG
from src.dataset import make_toy_loaders
from src.engine import run_epoch
from src.model import TinyClassifier

train_loader, val_loader = make_toy_loaders(batch_size=CONFIG["batch_size"], seed=CONFIG["seed"])
model = TinyClassifier(hidden_dim=CONFIG["hidden_dim"])
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=CONFIG["lr"])

history = []
for epoch in range(1, CONFIG["epochs"] + 1):
    train_loss, train_acc = run_epoch(model, train_loader, loss_fn, optimizer=optimizer)
    val_loss, val_acc = run_epoch(model, val_loader, loss_fn, optimizer=None)
    history.append((epoch, train_loss, train_acc, val_loss, val_acc))
    print(
        f"modular epoch={epoch:02d} | train_loss={train_loss:.4f} | train_acc={train_acc:.4f} | "
        f"val_loss={val_loss:.4f} | val_acc={val_acc:.4f}"
    )

print("final history row =", history[-1])

## 3. 什么时候该抽模块 / When Should You Extract Modules?

一个很实用的判断标准 / A practical rule of thumb:

- 同一段代码复制了两次以上 / the same code has been copied more than once
- 训练流程已经超过你一眼能看懂的长度 / the training workflow is longer than what you can comfortably understand at a glance
- 你需要多次重复运行实验 / you need to rerun experiments many times
- 你需要向别人解释项目结构 / you need to explain the project structure to someone else

In [ ]:
# 练习 1 / Exercise 1
# 如果你写了一个自定义 Dataset 类，它最适合放在哪个文件？
# If you write a custom Dataset class, which file is the best place for it?
#
# A. model.py
# B. dataset.py
# C. train.py

练习 1 参考答案 / Exercise 1 Reference Answer

- 正确答案是 `B. dataset.py` / The correct answer is `B. dataset.py`

因为它的职责是准备数据，而不是定义模型或组织训练流程。  
Because its responsibility is data preparation, not model definition or training orchestration.

In [ ]:
# 练习 2 / Exercise 2
# 下面这些内容分别更适合放哪？
# Which file is each of the following most suitable for?
#
# 1. nn.Module 子类 / an nn.Module subclass
# 2. train_one_epoch 函数 / a train_one_epoch function
# 3. batch_size 和 learning rate / batch_size and learning rate

练习 2 参考答案 / Exercise 2 Reference Answer

1. `model.py`
2. `engine.py` 或 `train_utils.py`
3. `config.py` 或配置文件 / `config.py` or a config file

这类划分不是唯一标准，但要尽量保持职责稳定。  
This is not the only valid split, but the responsibilities should stay stable.

## 4. 小结 / Summary

这一节最重要的收获 / The most important takeaways from this notebook are:

1. notebook 原型很适合探索，但不适合长期维护 / notebook prototypes are great for exploration, but not ideal for long-term maintenance
2. `dataset / model / engine / config` 这套拆分非常常见 / the `dataset / model / engine / config` split is very common
3. 把代码抽成模块之后，训练流程更清晰、更容易复用 / after extracting modules, the training workflow becomes clearer and more reusable
4. 你不需要一开始就工程化，但当项目开始重复和变复杂时，就应该抽模块 / you do not need to over-engineer from day one, but once the project becomes repetitive and more complex, you should extract modules